## Imports

In [1]:
import sys
sys.path.insert(0, '..')

import torch
import pandas as pd
from nucb_transformer.utils.io import load_checkpoint, best_checkpoint_path
from nucb_transformer.training.metrics import ACTIVITY_CLASSES, confusion_matrix_df
from scripts.predict import predict_sequences

In [2]:
device = 'cpu'

In [3]:
# Load the best checkpoint from the default directory
device = torch.device(device)
ckpt_path = best_checkpoint_path('../checkpoints/')
model, _ = load_checkpoint(ckpt_path, device=device)
print(f'Loaded: {ckpt_path}')

Loaded: ../checkpoints/epoch_044_acc0.6746.pt


## Explore data

In [8]:
# load all data
from nucb_transformer.data.dataset import load_landscape
df = load_landscape()
df.head()

# training data = mutations < 2 and test data = mutations >= 2
train_df = df[df['num_mutations'] < 2]
test_df = df[df['num_mutations'] >= 2]

In [9]:
WT_seq = train_df[train_df['num_mutations'] == 0]['sequence'].values[0]

## Predict on a few sequences

In [ ]:
# # Can do multiple predictions at once
# sequences = [
#     'SEQUENCE_1',
#     'SEQUENCE_2',
# ]

sequence =  WT_seq

result = predict_sequences(model, [sequence], device)
result

,sequence,predicted_class,prob_activity > 0,prob_activity > A73R,prob_activity > WT,prob_non-functional
0,MIKKWAVHLLFSALVLLGLSGGAAYSPQHAEGAARYDDVLYFPASR...,non-functional,0.305978,0.012942,0.176585,0.504495


## Load and explore test set predictions
Run `python scripts/evaluate.py --save_predictions results/test_predictions.csv` first.

In [11]:
test_df = pd.read_csv('../results/test_predictions.csv')
test_df.head()

,mutations,num_mutations,sublibrary_names,generations,activity_level,is_functional,sequence,predicted_class,correct
0,"(('A', 30, 'S'), ('D', 91, 'A'), ('E', 94, 'A'...",4,"('g1_eppcr',)","('g1',)",non-functional,False,MIKKWAVHLLFSALVLLGLSGGAAYSPQHSEGAARYDDVLYFPASR...,non-functional,True
1,"(('A', 30, 'T'), ('D', 91, 'A'), ('E', 94, 'A'...",4,"('g1_eppcr',)","('g1',)",non-functional,False,MIKKWAVHLLFSALVLLGLSGGAAYSPQHTEGAARYDDVLYFPASR...,non-functional,True
2,"(('A', 33, 'C'), ('A', 34, 'D'), ('D', 38, 'L'...",7,"('prosar+screen_g2_redux',)","('g4',)",non-functional,False,MIKKWAVHLLFSALVLLGLSGGAAYSPQHAEGCDRYDLVLYFPASR...,activity > WT,False
3,"(('A', 33, 'C'), ('A', 34, 'E'), ('S', 45, 'K'...",5,"('g3_prosar_low_unscreened',)","('g3',)",non-functional,False,MIKKWAVHLLFSALVLLGLSGGAAYSPQHAEGCERYDDVLYFPAKR...,non-functional,True
4,"(('A', 33, 'C'), ('A', 34, 'V'), ('R', 46, 'Q'...",20,"('g4_other',)","('g4',)",non-functional,False,MIKKWAVHLLFSALVLLGLSGGAAYSPQHAEGCVRYDDVLYFPASQ...,activity > WT,False


In [12]:
# Accuracy by number of mutations
test_df.groupby('num_mutations')['correct'].mean().rename('accuracy')

num_mutations
3     0.580635
4     0.480538
5     0.401337
6     0.347127
7     0.312396
8     0.340445
9     0.356112
10    0.351020
11    0.296514
12    0.239753
13    0.109677
14    0.105263
15    0.149635
16    0.038961
17    0.032258
18    0.000000
19    0.000000
20    0.026490
21    0.040000
22    0.000000
23    0.000000
Name: accuracy, dtype: float64

In [ ]:
# High-activity variants the model missed
missed = test_df[
    test_df['activity_level'].isin(['activity > WT', 'activity > A73R']) &
    ~test_df['correct']
]
missed[['sequence', 'num_mutations', 'activity_level', 'predicted_class']]

In [13]:
# Confusion matrix
import numpy as np
from nucb_transformer.training.metrics import ACTIVITY_CLASSES

label_to_idx = {c: i for i, c in enumerate(ACTIVITY_CLASSES)}
preds_np  = test_df['predicted_class'].map(label_to_idx).values
labels_np = test_df['activity_level'].map(label_to_idx).values

confusion_matrix_df(preds_np, labels_np)

,activity > 0,activity > A73R,activity > WT,non-functional
activity > 0,1187,193,4620,1516
activity > A73R,26,6,138,19
activity > WT,1138,273,6885,1061
non-functional,4084,576,11107,9865
